# OCR a ticker's filings — **one notebook, two machines**

Edit **cell 1** and run top to bottom. `ENVIRONMENT` is the only thing that decides where the
OCR happens; everything else means the same in both.

| `ENVIRONMENT` | the parse runs | the statement CSVs are written |
|---|---|---|
| `"LOCAL"` | on this machine's card (RTX 3050) | **as each quarter finishes** |
| `"KAGGLE"` | on a Kaggle T4, through `kgpu` | once, when the run folder is pulled back |

⚠️ **THE T4 IS NOT FASTER THAN THIS LAPTOP.** Measured twice, interleaved: local
**0.62-0.68 s/page** against T4 **0.78-0.95**. What Kaggle buys is a *second machine running in
parallel, free, without occupying this one* — that is worth having and it is not a multiplier.
CLAUDE.md §6-2-duodetricies.

## What an interrupted run leaves behind

With `MERGE_INTO_CSV = True` a **LOCAL** run upserts each quarter into
`raw_data/cafef/financials/statements/` the moment that quarter finishes, through
`pdf_ocr_merge` and its three refusals. `FinancialsBuilder._write` renders to a `.tmp` and
`os.replace`s it, and only the quarters a merge PRODUCED are rewritten — so stopping a 12-hour
run at hour 6 keeps every quarter that finished, unchanged, and can lose at most the one in
flight. A backup of all three CSVs is taken before the first write.

⚠️ **ON KAGGLE THAT GUARANTEE IS THE PULL's, NOT THE RUN's.** A kernel writes `/kaggle/working`
and exits; there is no path from it to this disk. The worker still writes one JSON per filing
as it goes, but the CSVs here are only touched after `kgpu pull` brings the folder home.

## ⚠️ Three things this cannot do for you

1. **A statement a worker accepts is not always one a full run would.** `sane`'s magnitude band
   is reconstructed from the `pdf` rows on DISK here (`seed_history`) and accumulated IN THE RUN
   by `FinancialsBuilder.build()` — two populations, so the gate can disagree with itself.
   Measured on VIC Q3-2014.
2. ⚠️ **A TICKER WITH NOTHING ON DISK YET HAS NO BAND AT ALL, SO WITHOUT
   `FORCE_EMPTY_BAND` A GREEN RUN CREATES NO CSV.** Its first run is unguarded by
   construction and every merge of it is refused, so nothing is written, so the band is
   still empty next time — `BND-1` closes on itself. Measured on HOSE_BSR, 2026-08-30: a
   14-document Kaggle run finished clean, wrote a full run folder and not one statement
   CSV. `FORCE_EMPTY_BAND = True` breaks that loop for a ticker's FIRST run and lifts no
   other refusal; what it costs is the guard, so screen the artefact (the unit per report,
   total assets quarter on quarter) before quoting anything from a bootstrap run. The
   authoritative path for a new name is still a full Dagster `raw/cafef_financials` run,
   which accumulates its band as it goes.
3. ⚠️ **Nothing from a non-bank template may be quoted as a fundamental yet** — `CRP-1`.

In [9]:
# ── PARAMETERS — the only cell you edit ───────────────────────────────────────
ENVIRONMENT = "KAGGLE"        # "LOCAL" = parse here | "KAGGLE" = ship it to a T4
EXCHANGE    = "HOSE"         # HOSE | HNX | UPCOM
SYMBOL      = "TCB"          # ticker, as CafeF files it

# WHICH QUARTERS — YYYY-QQ. "2026-Q4" and the zero-padded "2026-04" are the same quarter.
#   ["2014-Q3"]              -> one quarter
#   ["2013-Q4", "2014-01"]   -> a batch, in any order
#   []  or  None             -> EVERY quarter this ticker files   (⚠️ 70+ documents, hours)
# ⚠️ The repo-native "Q3-2014" is REFUSED rather than quietly accepted: a typo has to report
#    itself as a typo, not as a quarter CafeF does not file.
QUARTERS = ["2013-Q3"]        # ["2021-01"] is the same quarter, written the other way

# ⚠️ OVERWRITE decides two things, and they are one question: what do we do about a quarter
#    that is already on disk?
#   False -> FILL THE GAPS. A quarter already reading `pdf` in all three statements is dropped
#            before any OCR (and before it is uploaded), and a figure that DIFFERS from a good
#            `pdf` row is never written over it.
#   True  -> re-parse every selected quarter and let the result replace what disk holds.
OVERWRITE = False

# UPSERT the accepted statements into raw_data/.../statements/*.csv.
#   LOCAL  -> one quarter at a time, as each finishes. This is the interruption guarantee.
#   KAGGLE -> once, after the pull. A kernel has no path to this disk.
# Either way it goes through `pdf_ocr_merge`, which BACKS THE THREE CSVs UP FIRST, prints every
# changed cell, and refuses FOUR things it cannot judge: a cumulative income statement, a
# statement whose `sane` band was empty, a figure that DIFFERS from a good `pdf` row, and
# ⚠️ a document any of whose layers RAISED — an exception measures the MACHINE, not the
# filing, so whatever won the cascade won by default (`VCR-1`, 2026-08-29).
MERGE_INTO_CSV = True

# ⚠️ BOOTSTRAP A TICKER THAT HAS NO STATEMENT CSV YET — and it lifts a real guard.
#    `sane`'s magnitude band is rebuilt from the `pdf` rows ALREADY ON DISK
#    (`seed_history`), so a ticker parsed for the first time has NO band, `sane` fails
#    open, and the merge then refuses every statement the run produced. Nothing is
#    written, so the band stays empty and the next run refuses again — the loop closes
#    on itself. That is `BND-1`, and it is why a run can finish green and create no CSV.
#   True  -> write those statements anyway. This is the ONLY way a new ticker is ever
#            bootstrapped. The other three refusals are untouched: a cumulative income
#            statement, a figure that DIFFERS from a good `pdf` row, and a document
#            whose engine RAISED are still skipped and SAID.
#   False -> keep the guard. Correct for a ticker that already has history on disk,
#            where the band is real and a run that trips it is telling you something.
# ⚠️ What it costs: those figures passed NO magnitude check. Screen the artefact
#    before quoting any of them — the unit per report, and total assets quarter on
#    quarter. Both are read off the run folder alone: no PDF, no OCR, no network.
FORCE_EMPTY_BAND = True

TEMPLATE     = None      # None = RESOLVE it (templates.csv, then CafeF's fingerprint). ⚠️ Never defaulted to "bank".
ALLOW_PARENT = False     # fall back to the STANDALONE filing where no consolidated one exists
PERIODS      = None      # the repo-native form, e.g. ["Q3-2014"]. Optional, and INTERSECTS with QUARTERS.
LAYERS       = None      # None = the full 47-layer cascade, in cascade order
COMPARE      = True      # score every parsed cell against the statement CSV already on disk
NOTES        = ""        # free text into the run folder; blank writes a sensible default

EXECUTE  = True          # False = resolve and print the plan, spend nothing
REHEARSE = True          # KAGGLE only: the worker side, locally, no quota (~60 s)

## Resolving the inputs

Nothing below is edited. The cells validate the parameters, build the **one** input object the
CLI, this notebook and `kgpu` all share (`pdf_ocr_job.JobSpec`), and print what would run
before anything is spent.

In [10]:
# ── SETUP — validate the parameters and find the repo ─────────────────────────
# ⚠️ Checked HERE, before a payload is built or a page is rendered: every one of these is a
# mistake that would otherwise surface hours later, or as a Kaggle round trip.
import os
import sys
from pathlib import Path

ENVIRONMENT = str(ENVIRONMENT).upper()
EXCHANGE = str(EXCHANGE).upper()
SYMBOL = str(SYMBOL).upper()
if ENVIRONMENT not in ("LOCAL", "KAGGLE"):
    raise ValueError(f"ENVIRONMENT must be 'LOCAL' or 'KAGGLE', not {ENVIRONMENT!r}")
if EXCHANGE not in ("HOSE", "HNX", "UPCOM"):
    raise ValueError(f"EXCHANGE must be HOSE, HNX or UPCOM, not {EXCHANGE!r}")

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "kaggle_gpu").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"no src/kaggle_gpu at or above {Path.cwd()} — open this notebook "
                       f"from inside the repo.")
# ⚠️ `kgpu` stages the payload and talks to the Kaggle client relative to the CWD, so the
# notebook anchors itself the way a shell would. LOCAL does not need it and gets it anyway:
# one behaviour, printed, beats two that differ by a mode.
os.chdir(REPO / "src" / "kaggle_gpu")
for _p in (REPO / "src", REPO / "src" / "kaggle_gpu"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from web_scraper import pdf_ocr_job as job          # noqa: E402

# ⚠️ FOLDED ONCE, HERE. "2026-04" and "2026-Q4" are one quarter, and the job name, the payload
# directory and the Kaggle kernel slug are all derived from this list — two spellings that
# reached those would be two runs racing for one slug. `None` means every quarter.
QUARTERS = job.canonical_quarters(QUARTERS)

print(f"environment : {ENVIRONMENT}")
print(f"ticker      : {EXCHANGE}_{SYMBOL}")
print(f"quarters    : {QUARTERS or 'ALL — every quarter this ticker files'}")
print(f"overwrite   : {OVERWRITE}"
      + ("" if OVERWRITE else "   (quarters already `pdf` in all three are skipped)"))
print(f"upsert csv  : {MERGE_INTO_CSV}"
      + ("   per quarter, as each finishes" if MERGE_INTO_CSV and ENVIRONMENT == "LOCAL"
         else "   after the pull" if MERGE_INTO_CSV else ""))
print(f"bootstrap   : {FORCE_EMPTY_BAND}"
      + ("   an EMPTY `sane` band is written anyway — the only way a new ticker "
         "starts" if FORCE_EMPTY_BAND else "   an EMPTY `sane` band is REFUSED"))
print(f"repo        : {REPO}")
print(f"cwd         : {Path.cwd()}")

environment : KAGGLE
ticker      : HOSE_TCB
quarters    : ['2013-Q3']
overwrite   : False   (quarters already `pdf` in all three are skipped)
upsert csv  : True   after the pull
repo        : d:\GIT\master-thesis
cwd         : d:\GIT\master-thesis\src\kaggle_gpu


In [11]:
# ── THE JOB — resolved and printed, before anything is spent ──────────────────
# ⚠️ Both branches end at the SAME object. `pdf_ocr.job()` writes a `JobSpec`'s fields into the
# worker notebook's parameter cell, and the worker builds the JobSpec from them — so a LOCAL
# run and a KAGGLE run of the same parameters are the same procedure on two machines, not two
# procedures. What differs is the stack, and every run records its `stack_fingerprint`.
SPEC = CFG = PREPARED = None

if ENVIRONMENT == "LOCAL":
    SPEC = job.JobSpec(
        exchange=EXCHANGE, symbol=SYMBOL, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare_with_disk=COMPARE, merge_into_csv=MERGE_INTO_CSV,
        force_empty_band=FORCE_EMPTY_BAND,
        notes=NOTES or f"ENVIRONMENT=LOCAL overwrite={OVERWRITE}",
    )
    # ⚠️ `prepare()` resolves the data root, the models, the TEMPLATE and the document list and
    # RAISES on any of them — no OCR, no PDF. It also raises, in as many words, when every
    # quarter you asked for is already parsed and OVERWRITE is False.
    PREPARED = SPEC.prepare()
    print("\n".join(PREPARED.describe()))
    print()
    for _t in PREPARED.tasks:
        print(f"  {_t.period:<8} {_t.file[:56]:<56} "
              f"{os.path.getsize(_t.path) / 1024 ** 2:>6.1f} MB"
              + ("  CUMULATIVE" if _t.cumulative else ""))
    if PREPARED.template != "bank":
        print(f"\n⚠️ CRP-1: this is a `{PREPARED.template}` filing. `C_LIABILITIES` still "
              f"misses on corp,\n   so the balance sheet reconciles on the TRIVIAL "
              f"`assets == resources` — true by\n   construction on any page that reads both. "
              f"Nothing from a non-bank run may be\n   quoted as a fundamental yet.")
else:
    from kgpu import pdf_ocr, runner                 # noqa: E402

    CFG = pdf_ocr.job(
        SYMBOL, exchange=EXCHANGE, periods=PERIODS, quarters=QUARTERS,
        allow_parent=ALLOW_PARENT, overwrite=OVERWRITE, template=TEMPLATE, layers=LAYERS,
        compare=COMPARE, notes=NOTES, merge_statements=MERGE_INTO_CSV,
        # ⚠️ NOT a worker parameter. The worker cannot upsert — it writes
        # /kaggle/working and exits — so this is the PULL's knob, read by
        # `runner.merge_statements` on this machine after the folder lands.
        force_empty_band=FORCE_EMPTY_BAND,
    )
    print("\n".join(pdf_ocr.describe(CFG)))
    print()
    # The filings this selects are the filings the WORKER will open: `plan()` runs HERE, so the
    # payload cannot diverge from the worker's own choice.
    runner.plan(CFG)

job          : pdf-ocr-tcb-2013-q3
kernel       : lyductrung/mt-pdf-ocr-tcb-2013-q3
dataset      : lyductrung/mt-cafef-filings-tcb-2013-q3
filings      : HOSE_TCB  periods=all  quarters=['2013-Q3']  allow_parent=False
template     : resolve on the worker
overwrite    : False   (quarters already `pdf` in all three statements are not shipped)
results into : reports/pdf_ocr

job          : pdf-ocr-tcb-2013-q3
kernel       : lyductrung/mt-pdf-ocr-tcb-2013-q3   [NvidiaTeslaT4]
notebook     : src/web_scraper/RUN__pdf_ocr.ipynb
dataset      : lyductrung/mt-cafef-filings-tcb-2013-q3
  ticker     : TCB
  tables     : 
  source     : src/web_scraper, src/utils
  uploaded   : ⚠️ never — run `python -m kgpu data` first
results into : reports/pdf_ocr
parameters   : 12 patched in place
  ALIGN_TORCH = False  # ⚠️ install torch 2.5.1+cu121 to match this machine — ~2.5 GB per Kaggle run
  MODE = 'kgpu'  # "auto" | "local" | "kgpu" — auto prints what it resolved to
  EXCHANGE = 'HOSE'
  SYMBOL = 'TCB'


In [12]:
# ── STAGE + REHEARSE — KAGGLE only: the worker side, locally, no quota ────────
# ⚠️ **THE PAYLOAD IS STAGED HERE, AND IT HAS TO BE.** A rehearsal runs the worker against
# `.payload/<job>/`, so there is nothing to rehearse until that exists — and until 2026-08-29
# this cell called `rehearse` straight after `plan`, which raises `no staged payload` on every
# first run of a job and points at `python -m kgpu data <job>`, a command that cannot resolve
# a job this notebook COMPUTED rather than wrote into kaggle_config.json.
# `export` is local and free — it writes the zip, it does not upload; the RUN cell below
# re-exports and uploads (`refresh_data=True`), so nothing here commits you to anything.
#
# ⚠️ The rehearsal runs no OCR pass. What it proves is that the payload holds every input the
# parse reads, under BOTH of Kaggle's mount layouts, and it prints the magnitude band `sane`
# will get. **An empty band is the warning to stop for**: `sane` fails open without one, and
# that is the documented way a run writes a wrong figure (CLAUDE.md §6-2-octodecies).
if ENVIRONMENT == "KAGGLE" and REHEARSE:
    from kgpu import export                       # noqa: E402

    export.export(CFG)                            # -> .payload/<job>/  (no upload)
    runner.rehearse(CFG)
else:
    print("skipped" if ENVIRONMENT == "KAGGLE" else "LOCAL — nothing to rehearse")

  1 filing(s) of HOSE_TCB (Q3-2013)
  documents.zip: 20 files, 85.9 MB (91.8 MB uncompressed)
  source.zip: 37 files from src/web_scraper, src/utils
staged payload -> D:\GIT\master-thesis\src\kaggle_gpu\.payload\pdf-ocr-tcb-2013-q3  (86.2 MB total)

rehearsing pdf-ocr-tcb-2013-q3 — flat mount, source.zip

rehearsing pdf-ocr-tcb-2013-q3 — datasets/lyductrung/mt-cafef-filings-tcb-2013-q3 mount, source/ unpacked


In [13]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# ⚠️ Each document's JSON is written BEFORE the next one starts, and — LOCAL, with
# MERGE_INTO_CSV — each quarter is upserted into the statement CSVs before the next document is
# opened. Single filings here cost over half an hour; a run that kept its results in memory
# would lose every one of them to the first interrupt.
#
# ⚠️ Budget: a filing accepted at layer 1 is ~1 min, one that defeats the whole cascade was
# 26-33 min when measured, and Kaggle adds a ~5 min QUEUE before anything starts.
FOLDER = EXIT = None

if not EXECUTE:
    print("EXECUTE = False — the plan above is resolved and nothing was spent")
elif ENVIRONMENT == "LOCAL":
    FOLDER = job.run(SPEC)
    print(FOLDER)
else:
    if MERGE_INTO_CSV:
        print("MERGE_INTO_CSV is on: accepted statements are upserted into\n"
              "    raw_data/.../statements/ after the pull, with a backup taken first\n"
              "    and every changed cell printed.")
    # `refresh_data=True` re-exports and re-uploads the payload every time — correct, because
    # the filter above may have changed since the last run of this job.
    EXIT = runner.run(CFG, refresh_data=True)
    print(f"\nexit {EXIT}   (0 = COMPLETE and pulled)")

MERGE_INTO_CSV is on: accepted statements are upserted into
    raw_data/.../statements/ after the pull, with a backup taken first
    and every changed cell printed.
  1 filing(s) of HOSE_TCB (Q3-2013)
  documents.zip: 20 files, 85.9 MB (91.8 MB uncompressed)
  source.zip: 37 files from src/web_scraper, src/utils
staged payload -> D:\GIT\master-thesis\src\kaggle_gpu\.payload\pdf-ocr-tcb-2013-q3  (86.2 MB total)
creating dataset lyductrung/mt-cafef-filings-tcb-2013-q3 (private)

  dataset pending (HTTPError)
  dataset ready v1
dataset ready: https://www.kaggle.com/datasets/lyductrung/mt-cafef-filings-tcb-2013-q3 (version 1)
staged RUN__pdf_ocr.ipynb (24 KiB) -> D:\GIT\master-thesis\src\kaggle_gpu\.build
pushing lyductrung/mt-pdf-ocr-tcb-2013-q3 | accelerator: NvidiaTeslaT4
        data: lyductrung/mt-cafef-filings-tcb-2013-q3
pushed version 1
watch: https://www.kaggle.com/code/lyductrung/mt-pdf-ocr-tcb-2013-q3
baseline: none yet — no percentage until this job completes once
[  0.0 min]

## The result

`compare()` scores every parsed cell against the statement CSV on disk, so a verdict means:

| verdict | what it says |
|---|---|
| `REPRODUCED` | every cell, the winning **layer**, the unit and `publish_date` match disk |
| `DIFFERS` | one of those moved — the run names which, with both figures |
| `absent in this run` | the cascade refused the statement; the log below says why |
| `no pdf row on disk to compare against` | ⚠️ **a RECOVERY, not a reproduction** — nothing scored it |
| *(refused)* | a cumulative income statement is not scored against a de-cumulated row |

⚠️ **READ THE FIRST REFUSAL, NOT THE LAST.** A cascade's final refusal names the hardest path
tried, not the blocking defect — the label `fx not mapped` sent this repo down a wrong
diagnosis for two days, and six of the seven quarters blamed on it then parsed at a **strict**
layer with no FX change at all (CLAUDE.md §6-2-duovicies).

In [14]:
# ── READ THE RUN FOLDER ───────────────────────────────────────────────────────
# ⚠️ Read back from disk rather than from anything in memory, so this measures what a later
# reader would actually get. `metadata.json` already carries the whole scorecard in `results`.
import json                                          # noqa: E402

PATTERN = f"*__{EXCHANGE.lower()}_{SYMBOL.lower()}__pdf_ocr"
FOLDERS = sorted((REPO / "reports" / "pdf_ocr").glob(PATTERN), key=lambda p: p.name)
LATEST = FOLDERS[-1] if FOLDERS else None

if LATEST is None:
    print(f"no run folder matching {PATTERN}")
else:
    META = json.loads((LATEST / "metadata.json").read_text(encoding="utf-8"))
    inputs, ocr = META.get("inputs", {}), META.get("environment", {}).get("ocr", {})
    print(LATEST.name)
    print(f"  commit       : {META.get('git_commit')}")
    # ⚠️ §5 rule 2 at the artefact: a run folder written before schema v2 carries none of the
    # three fields below, and printing `None` for them would read as a VALUE rather than as
    # "this run predates the field". `schema_version` is what tells the two apart.
    V2 = META.get("schema_version", 1) >= 2
    OLDER = "— (schema v1: this run predates the field)"
    print(f"  filter       : quarters={inputs.get('quarters')}  "
          f"periods={inputs.get('periods')}  "
          f"overwrite={inputs.get('overwrite') if V2 else OLDER}")
    print(f"  skipped      : {inputs.get('skipped_already_parsed') if V2 else OLDER}")
    print(f"  upserted     : {inputs.get('merged_into_csv') if V2 else OLDER}"
          + (f"   backup={inputs.get('merge_backup')}" if V2 else ""))
    print(f"  template     : {inputs.get('template')}  ({inputs.get('template_how')})")
    # ⚠️ THE TWO OCR HALVES FAIL INDEPENDENTLY — detection is onnxruntime, recognition is torch
    # — so "the GPU was used" is two questions. `ORT-1` is a green run that was half on the CPU
    # because onnxruntime ADVERTISED a provider the session then could not create.
    print(f"  detection    : {(ocr.get('det_providers') or ['?'])[0]}"
          f"   (onnxruntime {ocr.get('onnxruntime')})")
    print(f"  recognition  : {ocr.get('recognizer_device')}")
    print(f"  stack        : {ocr.get('stack_fingerprint')}"
          + (f"   ⚠️ PIN VIOLATIONS: {ocr['pin_violations']}"
             if ocr.get("pin_violations") else ""))
    print()
    print(f"  {'period':10} {'report':18} {'layer':30} {'items':>5}  {'status':8} verdict")
    for r in META.get("results", []):
        print(f"  {r['period']:10} {r['report']:18} {(r['layer'] or '—'):30} "
              f"{r['items']:>5}  {r['status']:8} {r['verdict']}")
    # ⚠️ `seconds` is the DOCUMENT's cost repeated on each of its three report rows, so it is
    # summed per PERIOD. A set would also collapse two documents that took the same time.
    PER_DOC = {r["period"]: r["seconds"] for r in META.get("results", [])}
    print(f"\n  parse: {sum(PER_DOC.values()) / 60:.1f} min over {len(PER_DOC)} document(s)")

20260829-225620__hose_tcb__pdf_ocr
  commit       : f38efb2+dirty
  filter       : quarters=['2013-Q3']  periods=None  overwrite=False
  skipped      : []
  upserted     : False   backup=None
  template     : bank  (detect_template (CafeF fingerprint, over the network))
  detection    : CUDAExecutionProvider   (onnxruntime 1.22.0)
  recognition  : cuda
  stack        : 88df8ef02c08

  period     report             layer                          items  status   verdict
  Q3-2013    balance_sheet      —                                  0  absent   absent in this run
  Q3-2013    income_statement   —                                  0  absent   absent in this run
  Q3-2013    cash_flow          —                                  0  absent   absent in this run

  parse: 0.4 min over 1 document(s)


In [15]:
# ── WHAT WAS REFUSED, AND WHAT WAS WRITTEN ────────────────────────────────────
# The refusal reasons come from `_parse_cascaded`, which keeps one per layer and prints the
# DISTINCT ones with the first layer that gave each. The `WRITE`/`skip` lines are the upsert's
# own decisions, quarter by quarter.
if LATEST is not None:
    LOG = (LATEST / "run.log").read_text(encoding="utf-8", errors="replace")
    HITS = [ln for ln in LOG.splitlines()
            if "absent after" in ln or "reconcile:" in ln or "sane:" in ln
            or ln.lstrip().startswith(("WRITE ", "skip  ", "backup:", "written:"))]
    print("\n".join(HITS) if HITS else "no refusals — every statement was accepted")